In [35]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(19)

In [36]:
Q = torch.randn(3,4)
K = torch.randn(3,4)
V = torch.randn(3,4)

In [37]:
# Now let's compute self attention in matrix form
def attention(Q, K, V, mask=None):
  # trin 1: Dot product, we want words x words. We have two matrix of words x numbers.
  # the order is important as rows = queries and columns = keys
  # We have to define that its the last 2 dimension to transpose such that we always transpose the words and numbers. This was an error when heads and teksts were introduced.
  dot_product_matrix = Q @ K.transpose(-2,-1)
  print(dot_product_matrix.shape)

  # trin 2: scale by sqrt(dimension), we use Q.shape[-1] to make sure we get number of dimensions pr. word.
  scaled_dot_product = dot_product_matrix / math.sqrt(Q.shape[-1])

  # trin ekstra: mask, teksts have padding so they are the same length. this is important so the tensors have the same shape.
  # we do this, by making empty spaces = -infinty before the softmax, since softmax then makes them 0
  # This makes it so the words never pay attention to the padding. As softmax makes it 0%.
  # So the mask is a table with 1 if there is a word and 0 if this it padding.
  if mask is not None:
    scaled_dot_product = scaled_dot_product.masked_fill(mask == 0, -math.inf)

  # trin 3: softmax af attention, we use dim=-1 to always get the last dimension, which is keys.
  softmax_attention = torch.softmax(scaled_dot_product, dim=-1)
  print(softmax_attention)
  
  # trin 4: Every word becomes a weighted combinations of all the words values
  X_prime = softmax_attention @ V

  return X_prime, softmax_attention

In [38]:
##### TEST IF WORKS,  DELETE


# Run the attention function
X_prime, attention_weights = attention(Q, K, V)

# Check 1: form, skal være torch.Size([3, 4]): 3 ord, 4 tal pr. ord
print("Form af X_prime:", X_prime.shape)

# Check 2: hver række i attention skal summe til 1 (Notebook 12.1: "should be positive sum to one")
print("Rækkesummer:", attention_weights.sum(dim=-1))

# Selve resultatet
print(X_prime)

torch.Size([3, 3])
tensor([[0.2442, 0.5680, 0.1878],
        [0.0136, 0.0105, 0.9759],
        [0.0717, 0.3454, 0.5829]])
Form af X_prime: torch.Size([3, 4])
Rækkesummer: tensor([1., 1., 1.])
tensor([[-0.3819,  0.3114, -0.3864, -1.3563],
        [ 0.3922, -0.5987, -0.4263, -1.1151],
        [-0.1947, -0.0974, -0.4804, -1.2937]])


In [39]:
Q = torch.randn(2,2,3,4)
K = torch.randn(2,2,3,4)
V = torch.randn(2,2,3,4)

In [40]:
attention(Q,K,V)


torch.Size([2, 2, 3, 3])
tensor([[[[0.3645, 0.1752, 0.4604],
          [0.1796, 0.4034, 0.4170],
          [0.8902, 0.0535, 0.0563]],

         [[0.2767, 0.3314, 0.3919],
          [0.2787, 0.4206, 0.3007],
          [0.2565, 0.4820, 0.2615]]],


        [[[0.4718, 0.2482, 0.2800],
          [0.4117, 0.2276, 0.3607],
          [0.3827, 0.1936, 0.4236]],

         [[0.2052, 0.4422, 0.3526],
          [0.2439, 0.2815, 0.4747],
          [0.2695, 0.0625, 0.6679]]]])


(tensor([[[[-1.4224e+00,  4.2180e-02, -4.8640e-01,  1.1309e-01],
           [-1.6774e+00,  2.2424e-01, -3.2877e-01,  3.9410e-01],
           [-1.0032e+00, -1.1650e+00, -1.4773e+00,  7.0548e-01]],
 
          [[ 8.0004e-02, -2.1613e-01,  5.9423e-01,  8.7463e-01],
           [ 3.6740e-02, -2.8868e-01,  4.2074e-01,  9.3025e-01],
           [ 1.4970e-03, -3.5102e-01,  3.2851e-01,  9.7834e-01]]],
 
 
         [[[-2.2210e-01, -6.1666e-01, -3.2826e-01,  2.3611e-01],
           [-1.9524e-01, -6.4997e-01, -3.2511e-01,  2.4856e-01],
           [-1.7023e-01, -6.7830e-01, -3.3453e-01,  2.5311e-01]],
 
          [[-4.1007e-01, -4.5338e-01, -1.0032e+00,  6.8031e-01],
           [-3.5835e-01, -6.2888e-01, -1.2444e+00,  7.9447e-01],
           [-2.6719e-01, -8.4301e-01, -1.5565e+00,  9.0539e-01]]]]),
 tensor([[[[0.3645, 0.1752, 0.4604],
           [0.1796, 0.4034, 0.4170],
           [0.8902, 0.0535, 0.0563]],
 
          [[0.2767, 0.3314, 0.3919],
           [0.2787, 0.4206, 0.3007],
           [0.25

In [41]:
X_prime, attention_weights = attention(Q,K,V)

print("Form af X_prime:", X_prime.shape)               # skal være (2, 2, 3, 4)
print("Rækkesummer:", attention_weights.sum(dim=-1))    # skal være lutter 1-taller

torch.Size([2, 2, 3, 3])
tensor([[[[0.3645, 0.1752, 0.4604],
          [0.1796, 0.4034, 0.4170],
          [0.8902, 0.0535, 0.0563]],

         [[0.2767, 0.3314, 0.3919],
          [0.2787, 0.4206, 0.3007],
          [0.2565, 0.4820, 0.2615]]],


        [[[0.4718, 0.2482, 0.2800],
          [0.4117, 0.2276, 0.3607],
          [0.3827, 0.1936, 0.4236]],

         [[0.2052, 0.4422, 0.3526],
          [0.2439, 0.2815, 0.4747],
          [0.2695, 0.0625, 0.6679]]]])
Form af X_prime: torch.Size([2, 2, 3, 4])
Rækkesummer: tensor([[[1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000]],

        [[1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000]]])


In [42]:
# Nu prøver vi en mask

mask = torch.tensor([[1,1,1],[1,1,0]])
# Her har vi en form på (2, 3), 2 tekster og 3 pladser.
# Men vi ved at attention har formen (2,2,3,3). Så vi bruger unsqueeze til at udligne den her forskel. Ligesom i week 6 notebook.

mask = mask.unsqueeze(1).unsqueeze(1)
X_prime, attention_weights = attention(Q, K, V, mask=mask)
print("Tekst 2, hoved 1:\n", attention_weights[1, 0])

torch.Size([2, 2, 3, 3])
tensor([[[[0.3645, 0.1752, 0.4604],
          [0.1796, 0.4034, 0.4170],
          [0.8902, 0.0535, 0.0563]],

         [[0.2767, 0.3314, 0.3919],
          [0.2787, 0.4206, 0.3007],
          [0.2565, 0.4820, 0.2615]]],


        [[[0.6553, 0.3447, 0.0000],
          [0.6440, 0.3560, 0.0000],
          [0.6640, 0.3360, 0.0000]],

         [[0.3169, 0.6831, 0.0000],
          [0.4642, 0.5358, 0.0000],
          [0.8117, 0.1883, 0.0000]]]])
Tekst 2, hoved 1:
 tensor([[0.6553, 0.3447, 0.0000],
        [0.6440, 0.3560, 0.0000],
        [0.6640, 0.3360, 0.0000]])


In [ ]:
class MultiHeadAttention(torch.nn.Module):
    def __init__(self, emb_dim, n_heads):
        # We have the variables from encoder.yaml

        # We here make sure not to mess with Pytorchs own init definition, by using super we run theirs first and then continue to ours. Like in the Prince notebook 11_2
        super(MultiHeadAttention, self).__init__()

        # emb_dim must be divisible by n_heads, because each head gets an equal share of the numbers per word. If not we wouyld get an error.
        assert emb_dim % n_heads == 0

        # By doing this we make sure the variable n_heads doesnt go losst after the init
        self.n_heads = n_heads
        # Same for the dimensions pr. head like we did in notebook 6.1
        self.head_dim = emb_dim // n_heads

        # Define a Q, V, K and Output layer with the the input number emb_dim and same output number emb_dim.
        # As we saw in notebook 6.1, and the pytorch generates starting values. 
        self.linear1 = nn.Linear(emb_dim, emb_dim) # Q
        self.linear2 = nn.Linear(emb_dim, emb_dim) # K
        self.linear3 = nn.Linear(emb_dim, emb_dim) # V
        self.linear4 = nn.Linear(emb_dim, emb_dim) # Output

    def forward(self, x, mask=None):
        # Hvor x har formen (B, N, emb_dim), B = Batch (Antal tekster), N = Number of tokens (Længden af teksten)
        
    
        Q = self.linear1(x)
        K = self.linear2(x)
        V = self.linear3(x)


        # Hent størrelserne fra x: B = antal tekster, N = antal ord (bruges i view nedenfor)
        B, N, _ = x.shape

        # Split the dimension into multiple heads
        # Så i stedet for (B, N, emb_dim) så er det nu (B, N, n_heads, head_dim)
        Q = Q.view(B, N, self.n_heads, self.head_dim)

        # For det matcher attention definition, og bliver på formen (B, n_heads, N, head_dim)
        Q = Q.transpose(1,2)

        # Samme for K & V
        K = K.view(B, N, self.n_heads, self.head_dim)
        K = K.transpose(1,2)
        
        V = V.view(B, N, self.n_heads, self.head_dim)
        V = V.transpose(1,2)

        # Making sure the dimensions fit what our attention is expecting, by adding two ekstra dimensions. So we have (B, 1, 1, N)
        if mask is not None:
            mask = mask.unsqueeze(1).unsqueeze(1)   

        # Defining outputs of our attention function
        # Out has shape: (B, n_heads, N, head_dim)
        out, attention_weights = attention(Q, K, V, mask)

        self.attention_weights = attention_weights

        # Nu har vi brugt det som opdelt i hoveder og vil gerne have det tilbage på formen som resten af modellen kan bruge.
        # Nu har out formen (B, n_heads, N, head_dim)
        # Så vi laver transpose og view tilbage igen.
        out = out.transpose(1,2) # Nu har out formen (B, N, n_heads, head_dim)
        out = out.contiguous() # fixes error before view som i notebook 6
        out = out.view(B, N, self.emb_dim) # sætter hovederne side om side igen, så formen er (B, N, emb_dim)

        
        return out






In [44]:
mha = MultiHeadAttention(8, 2)
print(mha)
print(mha.n_heads, mha.head_dim)

MultiHeadAttention(
  (linear1): Linear(in_features=8, out_features=8, bias=True)
  (linear2): Linear(in_features=8, out_features=8, bias=True)
  (linear3): Linear(in_features=8, out_features=8, bias=True)
  (linear4): Linear(in_features=8, out_features=8, bias=True)
)
2 4


In [45]:
x = torch.randn(2, 3, 8)              # 2 texts, 3 words, 8 numbers per word
mask = torch.tensor([[1, 1, 1],
                     [1, 1, 0]])      # text 2 has padding on the last spot

mha = MultiHeadAttention(8, 2)
out = mha(x, mask)

print("out:", out.shape)                                   # skal være (2, 3, 8): samme form som x
print("weights:", mha.attention_weights.shape)             # skal være (2, 2, 3, 3)
print("tekst 2, hoved 1:\n", mha.attention_weights[1, 0])  # sidste kolonne skal være 0

torch.Size([2, 2, 3, 3])
tensor([[[[0.3491, 0.3970, 0.2538],
          [0.1603, 0.4167, 0.4230],
          [0.3774, 0.3091, 0.3135]],

         [[0.4222, 0.2740, 0.3037],
          [0.3490, 0.3798, 0.2711],
          [0.2522, 0.4015, 0.3463]]],


        [[[0.4453, 0.5547, 0.0000],
          [0.4219, 0.5781, 0.0000],
          [0.5314, 0.4686, 0.0000]],

         [[0.6318, 0.3682, 0.0000],
          [0.4045, 0.5955, 0.0000],
          [0.4801, 0.5199, 0.0000]]]], grad_fn=<SoftmaxBackward0>)


AttributeError: 'MultiHeadAttention' object has no attribute 'emb_dim'